# 05. Final Model and Evaluation

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Where we are:** notebook 04 chose logistic regression, the 11 features, and an alert line of 60%. Every choice there was made on four fifths of the shipments. One fifth was locked away.

**Goal of this notebook:** open the locked test set once and answer the judging question directly: **does the model help someone make a better decision, and can we prove it?**

1. Train the final model.
2. Check it has learned patterns and not memorised shipments.
3. Score it on the test set: how many alerts are real, how many disruptions are caught.
4. Check the risk tiers and compare with the rule of thumb.
5. Show what drives its predictions.
6. Show which actions lower the risk, and which do not.
7. Stress it with noisy, wrong and missing input.
8. Save it for the app.

**The rule we follow:** nothing is tuned in this notebook. The model, features and alert line are fixed before the test set is opened, so the test numbers are a fair measure.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA, OCHRE = "#2a78d6", "#eb6834", "#1baf7a", "#a07a12"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

## 1. Load the data and rebuild the same split

The features and the 80 / 20 split are identical to notebook 04, with the same random seed, so the test set here is exactly the fifth that notebook 04 never touched.

In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split

clean = pd.read_csv(ROOT / "data" / "processed" / "aura_shipments.csv")

SEED = 42
TARGET = "Disruption_Occurred"

WEATHER_SEVERITY = {"Clear": 0, "Rain": 1, "Fog": 2, "Storm": 3, "Hurricane": 4}
TYPICAL_PACE = {"Air": 2.35, "Rail": 20.9, "Road": 17.9, "Sea": 41.5}   # hours per 1,000 km in clear weather
HIGH_RISK_SCORE = 6.0        # geopolitical risk at or above this counts as a warning sign
LOW_RELIABILITY = 0.70       # carrier reliability below this counts as a warning sign


def add_features(df):
    """Add the engineered columns. Uses only inputs known during transit."""
    out = df.copy()
    severity = out["Weather_Condition"].map(WEATHER_SEVERITY)     # blank if the weather is missing or unknown
    hours = out["Lead_Time_Days"] * 24
    typical = out["Transport_Mode"].map(TYPICAL_PACE)

    out["weather_severity"] = severity                            # 0 clear ... 4 hurricane
    out["severe_weather_flag"] = (severity >= 3).astype(float).where(severity.notna())
    out["pace_h_per_1000km"] = hours / (out["Distance_km"] / 1000)
    out["pace_vs_typical"] = out["pace_h_per_1000km"] / typical    # 1.0 is normal, 3.0 is three times slower
    out["red_flag_count"] = (
        (severity >= 3).astype(int)
        + (out["Geopolitical_Risk_Score"] >= HIGH_RISK_SCORE).astype(int)
        + (out["Carrier_Reliability_Score"] < LOW_RELIABILITY).astype(int)
    )
    # Exposure of cold-chain cargo. Shown to the coordinator, never given to the model.
    out["hours_to_go"] = hours - out["Transit_Duration_Hours"]
    out["cold_gap_hours"] = out["Buffer_of_Package"] - out["hours_to_go"]
    return out


# The six recorded inputs every feature is built from
RAW_INPUTS = ["Weather_Condition", "Geopolitical_Risk_Score", "Carrier_Reliability_Score",
              "Lead_Time_Days", "Distance_km", "Transport_Mode"]
RAW_NUMERIC = ["Geopolitical_Risk_Score", "Carrier_Reliability_Score", "Lead_Time_Days", "Distance_km"]
ENGINEERED = ["weather_severity", "severe_weather_flag", "pace_h_per_1000km", "pace_vs_typical", "red_flag_count"]
MODEL_NUMERIC = RAW_NUMERIC + ENGINEERED
MODEL_CATEGORICAL = ["Weather_Condition", "Transport_Mode"]
MODEL_FEATURES = MODEL_NUMERIC + MODEL_CATEGORICAL

# Decisions carried over from notebook 04. Nothing below changes them.
ALERT_LINE = 0.60
TIER_EDGES = [0, 0.30, 0.60, 0.80, 1.0001]
TIER_NAMES = ["Low (0 to 30%)", "Medium (31 to 60%)", "High (61 to 80%)", "Critical (above 80%)"]
MODEL_SETTINGS = dict(C=0.3, max_iter=3000)

data = add_features(clean)
work, test = train_test_split(data, test_size=0.20, stratify=data[TARGET], random_state=SEED)
work, test = work.reset_index(drop=True), test.reset_index(drop=True)

X_work, y_work = work[MODEL_FEATURES], work[TARGET]
X_test, y_test = test[MODEL_FEATURES], test[TARGET]

assert TARGET not in MODEL_FEATURES, "The outcome is in the feature list"
assert not set(work["Shipment_ID"]) & set(test["Shipment_ID"]), "A shipment is in both sets"

print(f"Working set : {len(work):,} shipments, {y_work.mean():.1%} disrupted")
print(f"Test set    : {len(test):,} shipments, {y_test.mean():.1%} disrupted")
print("No shipment appears in both sets, and the outcome is not used as an input.")

Working set : 4,000 shipments, 61.3% disrupted
Test set    : 1,000 shipments, 61.3% disrupted
No shipment appears in both sets, and the outcome is not used as an input.


## 2. Train the final model

The model is trained once on the whole working set, with the settings chosen in notebook 04.

One step is added in front of the model: if a number is missing, it is filled with the typical value for that column, and a missing or unknown text value is treated as "no information". The training data has no gaps, so this changes nothing here. It is there so that a shipment with a blank field does not crash the app. Section 8 tests how well that works.

**A second, smaller model is trained alongside it.** Notebook 03 showed that the pace of a shipment gives away the weather. A model that has always been handed the weather never learns to use that. So we also train a **fallback model** with every weather column removed. It uses the same model type and the same settings, and it is only used when the weather field is blank or holds a word the main model does not know.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FALLBACK_NUMERIC = RAW_NUMERIC + ["pace_h_per_1000km", "pace_vs_typical"]
FALLBACK_CATEGORICAL = ["Transport_Mode"]
FALLBACK_FEATURES = FALLBACK_NUMERIC + FALLBACK_CATEGORICAL


def build_pipeline(numeric=MODEL_NUMERIC, categorical=MODEL_CATEGORICAL):
    fill_numbers = Pipeline([("fill_missing", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    fill_text = Pipeline([("fill_missing", SimpleImputer(strategy="constant", fill_value="Unknown")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))])
    prepare = ColumnTransformer([("numeric", fill_numbers, numeric), ("categorical", fill_text, categorical)])
    return Pipeline([("prepare", prepare), ("model", LogisticRegression(**MODEL_SETTINGS))])


final_model = build_pipeline().fit(X_work, y_work)
fallback_model = build_pipeline(FALLBACK_NUMERIC, FALLBACK_CATEGORICAL).fit(work[FALLBACK_FEATURES], y_work)


def predict_risk(shipments):
    """Disruption risk for each shipment. Uses the fallback model where the weather is missing or unknown."""
    featured = add_features(shipments)
    weather_known = featured["weather_severity"].notna().to_numpy()
    risk = np.empty(len(featured))
    if weather_known.any():
        risk[weather_known] = final_model.predict_proba(featured.loc[weather_known, MODEL_FEATURES])[:, 1]
    if (~weather_known).any():
        risk[~weather_known] = fallback_model.predict_proba(featured.loc[~weather_known, FALLBACK_FEATURES])[:, 1]
    return risk


print("Main model    :", final_model.named_steps["model"], f"on {len(MODEL_FEATURES)} features")
print("Fallback model:", fallback_model.named_steps["model"], f"on {len(FALLBACK_FEATURES)} features (no weather)")

Main model    : LogisticRegression(C=0.3, max_iter=3000) on 11 features
Fallback model: LogisticRegression(C=0.3, max_iter=3000) on 7 features (no weather)


## 3. Has it learned, or memorised?

A model that memorised its training shipments scores far better on them than on new ones. We compare three scores:

- **Seen:** the working set it trained on.
- **Unseen, inside the working set:** 5-fold cross-validation, as in notebook 04.
- **Test:** the locked fifth, opened now for the first time.

If the three are close, the model has learned patterns that carry over to new shipments.

In [4]:
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import cross_val_predict

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
p_seen = final_model.predict_proba(X_work)[:, 1]
p_unseen = cross_val_predict(build_pipeline(), X_work, y_work, cv=folds, method="predict_proba")[:, 1]
p_test = final_model.predict_proba(X_test)[:, 1]
p_test_fallback = fallback_model.predict_proba(test[FALLBACK_FEATURES])[:, 1]


def ranking_scores(truth, predicted):
    return {"roc_auc": roc_auc_score(truth, predicted),
            "avg_precision": average_precision_score(truth, predicted),
            "brier": brier_score_loss(truth, predicted)}


memory_check = pd.DataFrame({
    "Seen (working set)": ranking_scores(y_work, p_seen),
    "Unseen (cross-validation)": ranking_scores(y_work, p_unseen),
    "Test set": ranking_scores(y_test, p_test),
    "Test set, fallback model (no weather)": ranking_scores(y_test, p_test_fallback),
}).T
memory_check["shipments"] = [len(y_work), len(y_work), len(y_test), len(y_test)]
print(f"Gap between seen and test, ROC AUC          : {memory_check.loc['Seen (working set)', 'roc_auc'] - memory_check.loc['Test set', 'roc_auc']:+.3f}")
print(f"Gap between seen and test, average precision: {memory_check.loc['Seen (working set)', 'avg_precision'] - memory_check.loc['Test set', 'avg_precision']:+.3f}")
memory_check.round(4)

Gap between seen and test, ROC AUC          : +0.006
Gap between seen and test, average precision: +0.001


,roc_auc,avg_precision,brier,shipments
Seen (working set),0.8331,0.9023,0.1612,4000
Unseen (cross-validation),0.8301,0.9006,0.1625,4000
Test set,0.8269,0.9009,0.1638,1000
"Test set, fallback model (no weather)",0.7990,0.8787,0.1761,1000


**What this tells us:**

- **The three scores are almost the same.** ROC AUC is 0.833 on shipments the model trained on, 0.830 in cross-validation and 0.827 on the test set.
- **There is next to no gap.** A model this simple, with one weight per input column, has very little room to memorise individual shipments, and it shows.
- **Notebook 04 predicted the test result well.** Cross-validation said 0.830 and the test set gave 0.827, well inside the margin of error shown in the next section. The choices made in notebook 04 hold up on new shipments.
- **The fallback model scores 0.80 with no weather information at all.** It gives up about 3 points. Section 8 shows why that matters.

## 4. The scorecard on the test set

Every shipment at or above the 60% alert line is an alert. Four things can happen:

| | The shipment was disrupted | The shipment was fine |
|---|---|---|
| **Alert raised** | Caught in time | False alarm |
| **No alert** | Missed | Correctly left alone |

In [5]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

alert = p_test >= ALERT_LINE
truth = y_test.to_numpy()
left_alone, false_alarm, missed, caught = confusion_matrix(truth, alert).ravel()

outcome_table = pd.DataFrame(
    {"The shipment was disrupted": [caught, missed], "The shipment was fine": [false_alarm, left_alone]},
    index=["Alert raised", "No alert"],
)
print(f"Test shipments: {len(truth):,}   Disruptions: {truth.sum()}   Alerts raised: {alert.sum()}")
print()
print(f"Precision : {caught / (caught + false_alarm):.1%}   of {caught + false_alarm} alerts, {caught} were real")
print(f"Recall    : {caught / (caught + missed):.1%}   of {caught + missed} disruptions, {caught} were caught")
print(f"F1 score  : {f1_score(truth, alert):.3f}   (one number that balances precision and recall)")
print()
print(f"False alarms, model               : {false_alarm}")
print(f"False alarms, 'alert on everything': {int((truth == 0).sum())}   (precision {truth.mean():.1%}, recall 100%)")
print(f"Accuracy                          : {(caught + left_alone) / len(truth):.1%}   (shown for completeness, 'alert on everything' scores {truth.mean():.1%})")
outcome_table

Test shipments: 1,000   Disruptions: 613   Alerts raised: 508

Precision : 84.8%   of 508 alerts, 431 were real
Recall    : 70.3%   of 613 disruptions, 431 were caught
F1 score  : 0.769   (one number that balances precision and recall)

False alarms, model               : 77
False alarms, 'alert on everything': 387   (precision 61.3%, recall 100%)
Accuracy                          : 74.1%   (shown for completeness, 'alert on everything' scores 61.3%)


,The shipment was disrupted,The shipment was fine
Alert raised,431,77
No alert,182,310


### How sure are we about these numbers?

The test set has 1,000 shipments. A different 1,000 would give slightly different scores. To see how much, we redraw the test set 2,000 times at random (with repeats allowed) and re-score each time. The range that holds 95% of those results is the honest margin of error.

In [6]:
rng = np.random.default_rng(SEED)
draws = []
for _ in range(2000):
    idx = rng.integers(0, len(truth), len(truth))
    t, a, p = truth[idx], alert[idx], p_test[idx]
    draws.append({
        "Precision": precision_score(t, a, zero_division=0), "Recall": recall_score(t, a),
        "F1 score": f1_score(t, a), "ROC AUC": roc_auc_score(t, p), "Average precision": average_precision_score(t, p),
    })
draws = pd.DataFrame(draws)

scorecard = pd.DataFrame({
    "test_score": {
        "Precision": precision_score(truth, alert), "Recall": recall_score(truth, alert), "F1 score": f1_score(truth, alert),
        "ROC AUC": roc_auc_score(truth, p_test), "Average precision": average_precision_score(truth, p_test),
    },
    "low_95": draws.quantile(0.025), "high_95": draws.quantile(0.975),
})

fig = go.Figure(go.Scatter(
    x=scorecard["test_score"], y=scorecard.index, mode="markers+text",
    marker=dict(color=BLUE, size=12, line=dict(color=SURFACE, width=2)),
    text=[f"{v:.2f}" for v in scorecard["test_score"]], textposition="top center", textfont=dict(color=INK),
    error_x=dict(type="data", symmetric=False, array=scorecard["high_95"] - scorecard["test_score"],
                 arrayminus=scorecard["test_score"] - scorecard["low_95"], color=BLUE, thickness=1.5, width=5),
    hovertemplate="%{y}: %{x:.3f}<extra></extra>",
))
fig.update_xaxes(range=[0.5, 1.0], title_text="Score on the test set (bar = 95% range)")
fig.update_yaxes(autorange="reversed")
style(fig, "Test-set scorecard at the 60% alert line", height=380)
fig.update_layout(margin=dict(l=150))
fig.show()

scorecard.round(3)

,test_score,low_95,high_95
Precision,0.848,0.817,0.879
Recall,0.703,0.665,0.737
F1 score,0.769,0.740,0.796
ROC AUC,0.827,0.801,0.852
Average precision,0.901,0.883,0.918


**What this tells us:**

- **Precision is 85%.** Of 508 alerts, 431 were real disruptions and 77 were false alarms. The 95% range is 82% to 88%.
- **Recall is 70%.** Of 613 disruptions, 431 were caught before they happened and 182 were missed. The 95% range is 67% to 74%.
- **False alarms fall by four fifths.** Flagging every shipment would raise 387 false alarms. The model raises 77. The price is the 182 disruptions that fall below the line, and those stay visible further down the queue.
- **These match notebook 04** (85% and 69%). There is no drop on new shipments.
- **Accuracy is 74.1%**, against 61.3% for flagging everything. It is shown for completeness. The precision and recall figures are the ones that describe what a coordinator experiences.

**For the pitch:** "More than 8 in 10 alerts are real, and Aura catches 7 in 10 disruptions before they happen."

## 5. Risk tiers and the rule of thumb, on the test set

Two checks from notebook 04 are repeated here on shipments the model has never seen.

**Do the tiers still mean what they say?**

In [7]:
tier = pd.cut(p_test, bins=TIER_EDGES, labels=TIER_NAMES, right=False)
tiers = pd.DataFrame({"disrupted": truth}).groupby(tier, observed=False).agg(
    shipments=("disrupted", "size"), disrupted=("disrupted", "sum"), disruption_rate=("disrupted", "mean"))
tiers["share_of_all_disruptions"] = tiers["disrupted"] / tiers["disrupted"].sum()

fig = go.Figure(go.Bar(
    x=tiers.index.astype(str), y=tiers["disruption_rate"], marker_color=BLUE,
    text=[f"{r:.0%}<br>{n:,} shipments" for r, n in zip(tiers["disruption_rate"], tiers["shipments"])],
    textposition="outside", textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{x}<br>Really disrupted: %{y:.1%}<extra></extra>",
))
fig.update_yaxes(title_text="Share really disrupted", tickformat=".0%", range=[0, 1.22])
style(fig, "What really happened in each risk tier (test set)", height=440)
fig.update_layout(bargap=0.4)
fig.show()

tiers.style.format({"disruption_rate": "{:.1%}", "share_of_all_disruptions": "{:.1%}"})

,shipments,disrupted,disruption_rate,share_of_all_disruptions
Low (0 to 30%),155,34,21.9%,5.5%
Medium (31 to 60%),337,148,43.9%,24.1%
High (61 to 80%),207,140,67.6%,22.8%
Critical (above 80%),301,291,96.7%,47.5%


**Is the model still better than the rule of thumb?** The rule sorts shipments by how bad the weather is, then by geopolitical risk score. Both get the same number of alerts.

In [8]:
rule_score = (test["weather_severity"] * 100 + test["Geopolitical_Risk_Score"]).to_numpy()
rule_alert = np.zeros(len(truth), dtype=bool)
rule_alert[np.argsort(-rule_score, kind="stable")[:alert.sum()]] = True


def alert_quality(flag, label):
    hit = flag & (truth == 1)
    return {"method": label, "alerts": int(flag.sum()), "real": int(hit.sum()),
            "false_alarms": int((flag & (truth == 0)).sum()),
            "precision": hit.sum() / flag.sum(), "recall": hit.sum() / truth.sum()}


head_to_head = pd.DataFrame([alert_quality(rule_alert, "Rule of thumb"), alert_quality(alert, "Aura model")]).set_index("method")
fewer = 1 - head_to_head.loc["Aura model", "false_alarms"] / head_to_head.loc["Rule of thumb", "false_alarms"]
print(f"Same number of alerts ({alert.sum()}). The model raises {fewer:.0%} fewer false alarms than the rule of thumb.")

calm = test["Weather_Condition"].isin(["Clear", "Rain", "Fog"]).to_numpy()
low = p_test < 0.30
print()
print(f"Calm-weather shipments (clear, rain, fog): {calm.sum():,}, of which {truth[calm].mean():.1%} are disrupted")
print(f"  model alerts among them : {(alert & calm).sum()}, of which {truth[alert & calm].mean():.0%} are real")
print(f"  rated Low by the model  : {(low & calm).sum()}, of which {truth[low & calm].mean():.0%} are disrupted")
print(f"  ROC AUC, model against rule : {roc_auc_score(truth[calm], p_test[calm]):.3f} against {roc_auc_score(truth[calm], rule_score[calm]):.3f}")
head_to_head.style.format({"precision": "{:.1%}", "recall": "{:.1%}"})

Same number of alerts (508). The model raises 10% fewer false alarms than the rule of thumb.

Calm-weather shipments (clear, rain, fog): 601, of which 42.4% are disrupted
  model alerts among them : 123, of which 67% are real
  rated Low by the model  : 155, of which 22% are disrupted
  ROC AUC, model against rule : 0.672 against 0.592


,alerts,real,false_alarms,precision,recall
method,,,,,
Rule of thumb,508,422,86,83.1%,68.8%
Aura model,508,431,77,84.8%,70.3%


**Who does it work for?** Aura's user looks after cold-chain cargo, and a fifth of the file carries a quality flag. Here is the same scorecard for each group.

In [9]:
def group_scores(mask, label):
    hit = alert & mask & (truth == 1)
    return {"group": label, "shipments": int(mask.sum()), "disruptions": int(truth[mask].sum()),
            "alerts": int((alert & mask).sum()),
            "precision": hit.sum() / max((alert & mask).sum(), 1), "recall": hit.sum() / max(truth[mask].sum(), 1),
            "roc_auc": roc_auc_score(truth[mask], p_test[mask])}


cold = (test["Storage_standard"] != "Ambient").to_numpy()
pharma = (test["Product_Category"] == "Pharmaceuticals").to_numpy()
flagged = (test["Needs_Review"] == 1).to_numpy()

by_group = pd.DataFrame([
    group_scores(np.ones(len(test), dtype=bool), "All test shipments"),
    group_scores(cold, "Cold-chain cargo"),
    group_scores(pharma, "Pharmaceuticals only"),
    group_scores(~cold, "Ambient cargo"),
    group_scores(flagged, "Rows flagged as doubtful"),
    group_scores(~flagged, "Rows with no flag"),
]).set_index("group")

by_weather = test.assign(alert=alert, risk=p_test).groupby("Weather_Condition").agg(
    shipments=(TARGET, "size"), disruption_rate=(TARGET, "mean"), average_predicted_risk=("risk", "mean"),
    share_alerted=("alert", "mean")).loc[list(WEATHER_SEVERITY)]

display(by_group.style.format({"precision": "{:.0%}", "recall": "{:.0%}", "roc_auc": "{:.3f}"}))
by_weather.style.format({"disruption_rate": "{:.0%}", "average_predicted_risk": "{:.0%}", "share_alerted": "{:.0%}"})

,shipments,disruptions,alerts,precision,recall,roc_auc
group,,,,,,
All test shipments,1000,613,508,85%,70%,0.827
Cold-chain cargo,403,246,213,84%,73%,0.842
Pharmaceuticals only,212,128,113,82%,73%,0.845
Ambient cargo,597,367,295,85%,69%,0.817
Rows flagged as doubtful,178,92,74,78%,63%,0.775
Rows with no flag,822,521,434,86%,72%,0.836


,shipments,disruption_rate,average_predicted_risk,share_alerted
Weather_Condition,,,,
Clear,183,37%,36%,9%
Rain,199,44%,42%,16%
Fog,219,46%,50%,34%
Storm,197,79%,79%,93%
Hurricane,202,100%,99%,100%


**What this tells us:**

- **The tiers hold on new shipments.** About 22% of Low shipments were disrupted, 44% of Medium, 68% of High and 97% of Critical. Of 301 Critical shipments, 291 were disrupted.
- **The model still beats the rule of thumb, by a small margin.** With 508 alerts each, the model caught 431 disruptions and the rule caught 422, which means 10% fewer false alarms. A difference of 9 shipments in a test of 1,000 is small enough that chance plays a part. The larger working set in notebook 04 showed 5%. The direction is the same in both, and the exact size is uncertain.
- **It helps where the weather report is silent.** Among the 601 calm-weather shipments, 67% of the model's alerts were real against a base rate of 42%, and the 155 shipments it rated Low were disrupted only 22% of the time.
- **It works as well for cold-chain cargo as for anything else.** Precision is 84% and recall 73% on cold-chain shipments, and 82% and 73% on pharmaceuticals alone.
- **It is weaker on rows flagged as doubtful.** Precision is 78% and recall 63% there, against 86% and 72% on unflagged rows. Many flagged rows sit on the 0.5-day lead-time floor, where the pace cannot be trusted. There are only 178 of them, so the gap is uncertain, but the app should mark these shipments as "needs review" next to their risk.
- **The predicted risk tracks each weather closely.** The average prediction is within 4 points of what happened in every weather. Almost every storm shipment and every hurricane shipment is alerted, and fewer than 1 in 10 clear-weather shipments.

## 6. What drives the predictions?

A coordinator will not act on a number they do not understand. To see what the model relies on, we shuffle one recorded input at a time across the test shipments, rebuild the features, and measure how much worse the model gets. An input the model leans on causes a big drop. An input it ignores causes none.

We shuffle the six recorded inputs and not the eleven features, because several features are built from the same input. Shuffling the weather moves all three weather features together, which is the fair way to measure it.

In [10]:
INPUT_COLUMNS = RAW_INPUTS + ["Shipment_ID", "Place_of_Origin", "Place_of_Destination", "Product_Category",
                              "Storage_standard", "Buffer_of_Package", "Transit_Duration_Hours",
                              "Cargo_Value_USD", "Hospital_Urgency_Tier"]
test_inputs = test[INPUT_COLUMNS]

QUESTION = {
    "Weather_Condition": "How bad is the weather?",
    "Geopolitical_Risk_Score": "How risky is the route?",
    "Carrier_Reliability_Score": "How dependable is the carrier?",
    "Lead_Time_Days": "Is the trip running slow?",
    "Distance_km": "Is the trip running slow?",
    "Transport_Mode": "Is the trip running slow?",
}

rng = np.random.default_rng(SEED)
base_auc = roc_auc_score(truth, p_test)
rows = []
for col in RAW_INPUTS:
    drops = []
    for _ in range(30):
        shuffled = test_inputs.copy()
        shuffled[col] = rng.permutation(shuffled[col].to_numpy())
        drops.append(base_auc - roc_auc_score(truth, final_model.predict_proba(add_features(shuffled)[MODEL_FEATURES])[:, 1]))
    rows.append({"input": col, "question": QUESTION[col], "drop_in_roc_auc": np.mean(drops), "spread": np.std(drops)})
drivers = pd.DataFrame(rows).set_index("input").sort_values("drop_in_roc_auc")

fig = go.Figure(go.Bar(
    x=drivers["drop_in_roc_auc"], y=drivers.index, orientation="h", marker_color=BLUE,
    error_x=dict(type="data", array=drivers["spread"], color=INK, thickness=1, width=3),
    text=[f"{v:.3f}" for v in drivers["drop_in_roc_auc"]], textposition="outside", textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{y}<br>Drop in ROC AUC: %{x:.3f}<extra></extra>",
))
fig.update_xaxes(title_text="Drop in ROC AUC when this input is shuffled", range=[-0.01, drivers["drop_in_roc_auc"].max() * 1.25])
style(fig, f"How much the model leans on each input (test set, ROC AUC {base_auc:.3f} with nothing shuffled)", height=380)
fig.update_layout(margin=dict(l=200))
fig.show()

drivers.sort_values("drop_in_roc_auc", ascending=False).round(4)

,question,drop_in_roc_auc,spread
input,,,
Weather_Condition,How bad is the weather?,0.2355,0.0133
Geopolitical_Risk_Score,How risky is the route?,0.0582,0.0078
Lead_Time_Days,Is the trip running slow?,0.0426,0.0078
Transport_Mode,Is the trip running slow?,0.0244,0.0059
Carrier_Reliability_Score,How dependable is the carrier?,0.0091,0.0027
Distance_km,Is the trip running slow?,0.0057,0.0031


The chart says how much each input matters across all shipments. The next table says what each one does to a single shipment. We take a typical shipment (fog, risk score 5, carrier reliability 0.75, normal pace by sea) and change one thing at a time.

In [11]:
reference = pd.DataFrame([{
    "Weather_Condition": "Fog", "Geopolitical_Risk_Score": 5.0, "Carrier_Reliability_Score": 0.75,
    "Distance_km": 8000.0, "Transport_Mode": "Sea", "Transit_Duration_Hours": np.nan, "Buffer_of_Package": np.nan,
}])
SLOWDOWN = {"Clear": 1.0, "Rain": 1.2, "Fog": 1.5, "Storm": 3.0, "Hurricane": 8.0}   # measured in notebook 01 for surface modes


def with_weather(row, weather):
    """Set the weather and the lead time that goes with it for this mode and distance."""
    out = row.copy()
    out["Weather_Condition"] = weather
    out["Lead_Time_Days"] = TYPICAL_PACE[out["Transport_Mode"].iloc[0]] * SLOWDOWN[weather] * out["Distance_km"] / 1000 / 24
    return out


what_if = {"Typical shipment (fog, risk 5, reliability 0.75)": with_weather(reference, "Fog")}
for weather in WEATHER_SEVERITY:
    what_if[f"Weather: {weather}"] = with_weather(reference, weather)
for score in [1.0, 9.0]:
    what_if[f"Risk score: {score:.0f}"] = with_weather(reference, "Fog").assign(Geopolitical_Risk_Score=score)
for score in [0.55, 0.95]:
    what_if[f"Carrier reliability: {score}"] = with_weather(reference, "Fog").assign(Carrier_Reliability_Score=score)

one_at_a_time = pd.Series({label: predict_risk(row)[0] for label, row in what_if.items()}, name="predicted_risk").to_frame()
one_at_a_time["change_from_typical"] = one_at_a_time["predicted_risk"] - one_at_a_time["predicted_risk"].iloc[0]
one_at_a_time.style.format({"predicted_risk": "{:.0%}", "change_from_typical": "{:+.0%}"})

,predicted_risk,change_from_typical
"Typical shipment (fog, risk 5, reliability 0.75)",46%,+0%
Weather: Clear,32%,-14%
Weather: Rain,37%,-9%
Weather: Fog,46%,+0%
Weather: Storm,83%,+37%
Weather: Hurricane,100%,+54%
Risk score: 1,27%,-19%
Risk score: 9,69%,+23%
Carrier reliability: 0.55,54%,+8%
Carrier reliability: 0.95,41%,-5%


**What this tells us:**

- **The model mostly asks one question: how bad is the weather?** Shuffling the weather costs four times as much as shuffling anything else.
- **Then the route.** The geopolitical risk score is the second driver.
- **Lead time and transport mode matter through the pace.** The model expects the pace of a shipment to agree with its weather. Shuffle the lead time and a storm shipment appears to be moving at normal speed, which confuses it.
- **Carrier reliability and distance matter least.**

The one-at-a-time table puts sizes on this for a single shipment:

- **Weather** moves the typical shipment from 32% in clear weather to 83% in a storm and 100% in a hurricane.
- **Risk score** moves it from 27% on a calm route to 69% on a dangerous one.
- **Carrier reliability** moves it from 54% with a poor carrier to 41% with an excellent one.

For the app, this supports explaining each alert with three things a coordinator can check: the weather and how much it is slowing the shipment, the risk score of the route, and the reliability of the carrier.

## 7. Which actions lower the risk?

Prediction is only half of the product. The app lets a coordinator click an intervention and see what changes. Here we test that loop on every alert in the test set.

**Five actions, each changing an input the coordinator really controls**

| Action | What changes | Assumed cost |
|---|---|---|
| Replenish dry ice | 24 more hours of thermal buffer | $1,500 |
| Switch to a top-rated carrier | Carrier reliability raised to 0.95 | $2,000 |
| Reroute off the high-risk corridor | Geopolitical risk score capped at 3 | $5,000 |
| Carrier and reroute together | Both changes | $7,000 |
| Hold or route around the weather | Weather no worse than rain, and the lead time shortened to match | $8,000 |

The costs and the size of each effect are assumptions. The app shows them and lets the user change them.

**How money is counted:** the expected cost of a shipment is its predicted risk, times the share of cargo value a disruption costs (10%, as in notebook 04), times the cargo value. The saving from an action is the drop in expected cost minus the cost of the action.

In [12]:
LOSS_SHARE = 0.10   # assumption carried over from notebook 04: a disruption costs 10% of cargo value on average

ACTIONS = {
    "No action": ({}, 0),
    "Replenish dry ice": ({"buffer_hours": 24}, 1_500),
    "Switch to a top-rated carrier": ({"carrier_score": 0.95}, 2_000),
    "Reroute off the high-risk corridor": ({"risk_cap": 3.0}, 5_000),
    "Carrier and reroute together": ({"carrier_score": 0.95, "risk_cap": 3.0}, 7_000),
    "Hold or route around the weather": ({"weather_at_worst": "Rain"}, 8_000),
}


def apply_action(shipments, buffer_hours=0, carrier_score=None, risk_cap=None, weather_at_worst=None):
    """Change the recorded inputs the way the action would. Features are rebuilt when the risk is predicted."""
    changed = shipments[INPUT_COLUMNS].copy()
    changed["Buffer_of_Package"] = changed["Buffer_of_Package"] + buffer_hours
    if carrier_score is not None:
        changed["Carrier_Reliability_Score"] = changed["Carrier_Reliability_Score"].clip(lower=carrier_score)
    if risk_cap is not None:
        changed["Geopolitical_Risk_Score"] = changed["Geopolitical_Risk_Score"].clip(upper=risk_cap)
    if weather_at_worst is not None:
        worse = changed["Weather_Condition"].map(WEATHER_SEVERITY) > WEATHER_SEVERITY[weather_at_worst]
        speed_up = SLOWDOWN[weather_at_worst] / changed.loc[worse, "Weather_Condition"].map(SLOWDOWN)
        changed.loc[worse, "Lead_Time_Days"] = changed.loc[worse, "Lead_Time_Days"] * speed_up
        changed.loc[worse, "Weather_Condition"] = weather_at_worst
    return changed


alerted = test[alert].reset_index(drop=True)
risk_before = predict_risk(alerted)
cargo = alerted["Cargo_Value_USD"].to_numpy()

rows, risk_after = [], {}
for name, (change, cost) in ACTIONS.items():
    risk = predict_risk(apply_action(alerted, **change))
    risk_after[name] = risk
    saving = (risk_before - risk) * LOSS_SHARE * cargo - cost
    rows.append({
        "action": name, "average_risk": risk.mean(),
        "alerts_brought_under_the_line": int((risk < ALERT_LINE).sum()),
        "share_brought_under_the_line": (risk < ALERT_LINE).mean(),
        "median_net_saving_usd": np.median(saving),
        "share_where_action_pays_off": (saving > 0).mean(),
    })
actions = pd.DataFrame(rows).set_index("action")

# The app's job: for each shipment, recommend the action that saves the most, or none if nothing pays
savings = pd.DataFrame({name: (risk_before - risk_after[name]) * LOSS_SHARE * cargo - cost
                        for name, (_, cost) in ACTIONS.items()})
best_action = savings.idxmax(axis=1)
best_risk = np.array([risk_after[name][k] for k, name in enumerate(best_action)])
actions.loc["Best action for each shipment"] = {
    "average_risk": best_risk.mean(),
    "alerts_brought_under_the_line": int((best_risk < ALERT_LINE).sum()),
    "share_brought_under_the_line": (best_risk < ALERT_LINE).mean(),
    "median_net_saving_usd": savings.max(axis=1).median(),
    "share_where_action_pays_off": (savings.max(axis=1) > 0).mean(),
}
print("Which action is best, counted over the alerts:")
print(best_action.value_counts().to_string())

fig = go.Figure(go.Bar(
    x=actions.index, y=actions["average_risk"], marker_color=BLUE,
    text=[f"{r:.0%}" for r in actions["average_risk"]], textposition="outside",
    textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{x}<br>Average predicted risk: %{y:.1%}<extra></extra>",
))
fig.add_hline(y=ALERT_LINE, line=dict(color=MUTED, width=1, dash="dash"), layer="below")
fig.add_annotation(xref="paper", x=1, y=ALERT_LINE, text="Alert line 60%", showarrow=False,
                   xanchor="right", yanchor="bottom", font=dict(color=INK))
fig.update_yaxes(title_text="Average predicted risk", tickformat=".0%", range=[0, 1.05])
style(fig, f"Predicted risk across the {len(alerted)} test alerts, before and after acting", height=480)
fig.update_layout(bargap=0.45, margin=dict(b=140))
fig.show()

actions.style.format({"average_risk": "{:.0%}", "share_brought_under_the_line": "{:.0%}", "alerts_brought_under_the_line": "{:.0f}",
                      "median_net_saving_usd": "${:,.0f}", "share_where_action_pays_off": "{:.0%}"})

Which action is best, counted over the alerts:
No action                             223
Hold or route around the weather      215
Carrier and reroute together           63
Reroute off the high-risk corridor      5
Switch to a top-rated carrier           2


,average_risk,alerts_brought_under_the_line,share_brought_under_the_line,median_net_saving_usd,share_where_action_pays_off
action,,,,,
No action,85%,0,0%,$0,0%
Replenish dry ice,85%,0,0%,"$-1,500",0%
Switch to a top-rated carrier,81%,86,17%,"$-1,743",29%
Reroute off the high-risk corridor,74%,123,24%,"$-4,497",28%
Carrier and reroute together,70%,141,28%,"$-5,535",31%
Hold or route around the weather,46%,358,70%,"$-1,424",48%
Best action for each shipment,59%,244,48%,"$8,911",56%


The averages hide the most useful part. The same action does very different things depending on **why** the shipment is at risk. Here is the average risk after each action, split by the weather the alert is in.

In [13]:
by_cause = pd.DataFrame({name: pd.Series(risk).groupby(alerted["Weather_Condition"]).mean() for name, risk in risk_after.items()})
by_cause.insert(0, "alerts", alerted["Weather_Condition"].value_counts())
by_cause = by_cause.loc[[w for w in reversed(list(WEATHER_SEVERITY)) if w in by_cause.index]]

heat = by_cause.drop(columns="alerts")
fig = go.Figure(go.Heatmap(
    z=heat.values, x=heat.columns, y=[f"{w} ({n} alerts)" for w, n in zip(by_cause.index, by_cause["alerts"])],
    zmin=0, zmax=1, colorscale=[[0, "#f0efec"], [1, "#c8401f"]],
    text=heat.values, texttemplate="%{text:.0%}", textfont=dict(size=13),
    xgap=3, ygap=3, colorbar=dict(title="Risk", thickness=12, tickformat=".0%"),
    hovertemplate="%{y}<br>%{x}<br>Average predicted risk: %{z:.1%}<extra></extra>",
))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(tickangle=-25)
style(fig, "Average predicted risk after each action, by the weather the alert is in", height=440)
fig.update_layout(margin=dict(l=170, b=130))
fig.show()

# What the dry ice does change: thermal runway for cold cargo in passive packaging (air freight)
passive = alerted[(alerted["Storage_standard"] != "Ambient") & (alerted["Transport_Mode"] == "Air")]
gap_before = add_features(passive)["cold_gap_hours"]
gap_after = add_features(apply_action(passive, buffer_hours=24))["cold_gap_hours"]
print(f"Cold-chain alerts travelling by air, in passive packaging : {len(passive)}")
print(f"  buffer runs out before arrival, with no action          : {(gap_before < 0).sum()}")
print(f"  buffer runs out before arrival, after 24 more hours     : {(gap_after < 0).sum()}")
print(f"  predicted disruption risk for these, before and after   : {predict_risk(passive).mean():.0%} and {predict_risk(apply_action(passive, buffer_hours=24)).mean():.0%}")

Cold-chain alerts travelling by air, in passive packaging : 54
  buffer runs out before arrival, with no action          : 6
  buffer runs out before arrival, after 24 more hours     : 1
  predicted disruption risk for these, before and after   : 82% and 82%


### One shipment, start to finish

This is the story the demo tells. We take the pharmaceutical alerts in the test set and rank them the way the triage queue does: risk times cargo value times urgency weight, divided by the hours of thermal buffer left. Then we follow the top shipment that an action can still bring under the alert line.

In [14]:
URGENCY_WEIGHT = {"Routine Restock": 1.0, "Elevated Procedure": 1.5, "Critical Emergency": 2.5}

queue = alerted.assign(risk=risk_before)
queue = queue[queue["Product_Category"] == "Pharmaceuticals"].copy()
queue["priority"] = (queue["risk"] * queue["Cargo_Value_USD"] * queue["Hospital_Urgency_Tier"].map(URGENCY_WEIGHT)
                     / queue["Buffer_of_Package"].clip(lower=0.5))
best_single = np.minimum.reduce([risk_after[name] for name in ACTIONS if name != "No action"])
queue["can_be_brought_under_the_line"] = best_single[queue.index] < ALERT_LINE
queue = queue.sort_values("priority", ascending=False)

print(f"Pharmaceutical alerts in the test set: {len(queue)}, of which {int(queue['can_be_brought_under_the_line'].sum())} can be brought under the line by one action")
top = queue[queue["can_be_brought_under_the_line"]].iloc[0]
i = int(top.name)

print()
print(f"Shipment        : {top['Shipment_ID']}   {top['Place_of_Origin']}  ->  {top['Place_of_Destination']}  by {top['Transport_Mode'].lower()}")
print(f"Cargo           : ${top['Cargo_Value_USD']:,.0f}, {top['Storage_standard']}, {top['Hospital_Urgency_Tier']}")
print(f"Position        : number {list(queue.index).index(i) + 1} in the pharmaceutical queue")
print(f"Weather         : {top['Weather_Condition']}, {top['pace_vs_typical']:.1f} times the normal pace")
print(f"Route and carrier: risk score {top['Geopolitical_Risk_Score']:.1f}, carrier reliability {top['Carrier_Reliability_Score']:.2f}, {int(top['red_flag_count'])} red flags")
print(f"Thermal buffer  : {top['Buffer_of_Package']:.1f} hours left, {top['hours_to_go']:.0f} hours still to go")
print()
story = pd.DataFrame([
    {"action": name, "predicted_risk": risk_after[name][i],
     "expected_cost_usd": risk_after[name][i] * LOSS_SHARE * top["Cargo_Value_USD"],
     "action_cost_usd": cost,
     "net_saving_usd": (risk_before[i] - risk_after[name][i]) * LOSS_SHARE * top["Cargo_Value_USD"] - cost}
    for name, (_, cost) in ACTIONS.items()
]).set_index("action")
story.style.format({"predicted_risk": "{:.0%}", "expected_cost_usd": "${:,.0f}",
                    "action_cost_usd": "${:,.0f}", "net_saving_usd": "${:,.0f}"})

Pharmaceutical alerts in the test set: 113, of which 98 can be brought under the line by one action

Shipment        : SC-14451   Dubai  ->  Singapore  by air
Cargo           : $37,361,272, 2°C to 8°C, Critical Emergency
Position        : number 1 in the pharmaceutical queue
Weather         : Hurricane, 6.3 times the normal pace
Route and carrier: risk score 6.2, carrier reliability 0.68, 3 red flags
Thermal buffer  : 0.0 hours left, 9 hours still to go



,predicted_risk,expected_cost_usd,action_cost_usd,net_saving_usd
action,,,,
No action,99%,"$3,716,828",$0,$0
Replenish dry ice,99%,"$3,716,828","$1,500","$-1,500"
Switch to a top-rated carrier,99%,"$3,707,388","$2,000","$7,440"
Reroute off the high-risk corridor,99%,"$3,693,187","$5,000","$18,641"
Carrier and reroute together,98%,"$3,672,382","$7,000","$37,446"
Hold or route around the weather,55%,"$2,039,144","$8,000","$1,669,684"


**What this tells us:**

- **Dry ice does not lower the predicted risk at all.** It stays at 85%. This is correct, and it changes the demo. The model predicts whether a shipment will be disrupted, and more coolant does not stop a storm. What dry ice changes is how long the cargo survives the disruption. Of the 54 cold-chain alerts travelling by air, 6 would run out of buffer before arrival. With 24 more hours, 1 would. The file has no record of spoilage, so we cannot turn that into a risk percentage. **The app should show dry ice as hours of thermal runway gained, not as risk reduced.**
- **Changing the carrier or the route helps a little on average.** A top-rated carrier takes the average risk from 85% to 81%. Rerouting takes it to 74%. Doing both takes it to 70% and brings 28% of alerts under the line.
- **Holding or routing around the weather helps most.** The average falls to 46%, and 70% of alerts come under the line.
- **The right action depends on the cause, and the heatmap shows it.** For a hurricane alert, a better carrier or a safer route does nothing: the risk stays at 99%. Only getting out of the weather helps, taking it to about 41%. For an alert in fog, rain or clear weather it is the other way round. Those shipments are on the list because of their route and carrier, so rerouting cuts the risk roughly in half (from about 65% to about 35%), and the weather action does little or nothing.
- **Under our assumptions, no action is worth applying to every alert.** Each one pays for itself on fewer than half of the alerts, because a typical shipment is worth about $340,000 and a drop of a few points in risk is worth less than the action costs.
- **Choosing per shipment is what pays.** When the best action is picked for each alert, something pays off for 56% of them, with a median net saving of about $8,900. For the other 44% the best decision is to take no action and keep watching. Deciding better includes deciding not to spend.

**The example shipment** is SC-14451: $37 million of chilled pharmaceuticals, a Critical Emergency order, flying from Dubai to Singapore through a hurricane at six times the normal pace. Its risk is 99%. A better carrier or a safer corridor leaves it at 98 to 99%. Routing around the weather takes it to 55%, under the alert line, and is worth about $1.7 million in expected cost under our assumptions. Its thermal buffer is at zero with 9 hours still to go. Dry ice does not change the 99%, and it is still the first thing this shipment needs. Two different problems, two different actions.

**What this does and does not prove:** it shows that the model responds sensibly when an input changes, and in the direction a logistics expert would expect. It does not measure what an intervention achieves in the field. The model learned that shipments in calm weather on safe routes are disrupted less. It has never seen a shipment being moved from one to the other. The size of each effect and every cost are assumptions.

## 8. Stress test: noisy, wrong and missing input

Readings drift, ETAs are estimates, weather feeds lag, and forms are left half filled. The model has to stay useful when its inputs are imperfect.

### Noise in the numbers

We add random error to every numeric input of every test shipment, rebuild the features, and re-score. A noise level of 10% means each value is typically off by 10% in either direction. Each level is repeated 30 times and averaged.

In [15]:
def stress_scores(shipments):
    p = predict_risk(shipments)
    flag = p >= ALERT_LINE
    return {"roc_auc": roc_auc_score(truth, p), "precision": precision_score(truth, flag, zero_division=0),
            "recall": recall_score(truth, flag), "alerts": flag.sum()}


def with_noise(shipments, level, rng):
    noisy = shipments.copy()
    for col in RAW_NUMERIC:
        noisy[col] = noisy[col] * (1 + rng.normal(0, level, len(noisy)))
    noisy["Carrier_Reliability_Score"] = noisy["Carrier_Reliability_Score"].clip(0, 1)
    noisy["Geopolitical_Risk_Score"] = noisy["Geopolitical_Risk_Score"].clip(0, 10)
    noisy["Lead_Time_Days"] = noisy["Lead_Time_Days"].clip(lower=0.05)
    noisy["Distance_km"] = noisy["Distance_km"].clip(lower=50)
    return noisy


def with_wrong_weather(shipments, share, rng):
    """Replace the weather of a random share of shipments with a weather picked at random."""
    wrong = shipments.copy()
    pick = rng.random(len(wrong)) < share
    wrong.loc[pick, "Weather_Condition"] = rng.choice(list(WEATHER_SEVERITY), pick.sum())
    return wrong


rng = np.random.default_rng(SEED)
noise = pd.DataFrame([
    {"noise_level": level, **pd.DataFrame([stress_scores(with_noise(test_inputs, level, rng))
                                           for _ in range(1 if level == 0 else 30)]).mean().to_dict()}
    for level in [0.0, 0.02, 0.05, 0.10, 0.20, 0.30]
]).set_index("noise_level")
wrong_weather = pd.DataFrame([
    {"share_with_wrong_weather": share, **pd.DataFrame([stress_scores(with_wrong_weather(test_inputs, share, rng))
                                                        for _ in range(1 if share == 0 else 30)]).mean().to_dict()}
    for share in [0.0, 0.05, 0.10, 0.20, 0.30]
]).set_index("share_with_wrong_weather")

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                    subplot_titles=["Noise added to every numeric input", "Share of shipments with the wrong weather"])
for col, table in enumerate([noise, wrong_weather], start=1):
    for metric, label, color in [("roc_auc", "ROC AUC", BLUE), ("precision", "Precision", ORANGE), ("recall", "Recall", AQUA)]:
        fig.add_trace(go.Scatter(x=table.index, y=table[metric], mode="lines+markers", name=label, legendgroup=label,
                                 showlegend=col == 1, line=dict(color=color, width=2),
                                 marker=dict(size=8, line=dict(color=SURFACE, width=2)),
                                 hovertemplate=label + "<br>%{x:.0%}<br>%{y:.3f}<extra></extra>"), row=1, col=col)
    fig.update_xaxes(tickformat=".0%", tickvals=list(table.index), row=1, col=col)
fig.update_yaxes(range=[0.5, 1], title_text="Score on the test set", col=1)
fig.update_annotations(font=dict(size=12, color=INK))
style(fig, "How the scores hold up as the inputs get worse", height=440)
fig.update_layout(legend=dict(orientation="h", y=-0.2, x=0, font=dict(color=INK)), margin=dict(b=90))
fig.show()

display(noise.round(3))
wrong_weather.round(3)

,roc_auc,precision,recall,alerts
noise_level,,,,
0.00,0.827,0.848,0.703,508.000
0.02,0.827,0.846,0.698,505.567
0.05,0.826,0.844,0.697,505.700
0.10,0.824,0.838,0.693,507.267
0.20,0.821,0.835,0.692,508.167
0.30,0.811,0.819,0.695,520.267


,roc_auc,precision,recall,alerts
share_with_wrong_weather,,,,
0.00,0.827,0.848,0.703,508.000
0.05,0.816,0.840,0.697,508.233
0.10,0.804,0.830,0.691,510.633
0.20,0.781,0.814,0.674,507.333
0.30,0.757,0.794,0.666,514.267


### Missing fields

Now we blank out one input at a time for every test shipment, as if that field were never supplied. For the weather we show both what the main model does if left to cope, and what happens when the fallback model takes over.

In [16]:
def main_model_only(shipments):
    p = final_model.predict_proba(add_features(shipments)[MODEL_FEATURES])[:, 1]
    flag = p >= ALERT_LINE
    return {"roc_auc": roc_auc_score(truth, p), "precision": precision_score(truth, flag, zero_division=0),
            "recall": recall_score(truth, flag), "alerts": flag.sum()}


baseline = stress_scores(test_inputs)
missing_rows = [{"blank_field": "(nothing blank)", **baseline}]
for col in RAW_INPUTS:
    blanked = test_inputs.copy()
    blanked[col] = np.nan
    if col == "Weather_Condition":
        missing_rows.append({"blank_field": "Weather_Condition, main model left to cope", **main_model_only(blanked)})
        missing_rows.append({"blank_field": "Weather_Condition, fallback model takes over", **stress_scores(blanked)})
    else:
        missing_rows.append({"blank_field": col, **stress_scores(blanked)})

unknown_word = test_inputs.assign(Weather_Condition="Typhoon")
missing_rows.append({"blank_field": "Weather is a word the model has never seen ('Typhoon')", **stress_scores(unknown_word)})
no_weather_no_trip = test_inputs.assign(Weather_Condition=np.nan, Lead_Time_Days=np.nan)
missing_rows.append({"blank_field": "Weather and lead time both blank", **stress_scores(no_weather_no_trip)})

missing = pd.DataFrame(missing_rows).set_index("blank_field")
missing["roc_auc_lost"] = baseline["roc_auc"] - missing["roc_auc"]
missing.style.format({"roc_auc": "{:.3f}", "precision": "{:.0%}", "recall": "{:.0%}", "alerts": "{:.0f}", "roc_auc_lost": "{:+.3f}"})

,roc_auc,precision,recall,alerts,roc_auc_lost
blank_field,,,,,
(nothing blank),0.827,85%,70%,508,+0.000
"Weather_Condition, main model left to cope",0.677,71%,65%,560,+0.149
"Weather_Condition, fallback model takes over",0.799,83%,68%,498,+0.028
Geopolitical_Risk_Score,0.787,90%,58%,399,+0.039
Carrier_Reliability_Score,0.821,85%,68%,490,+0.006
Lead_Time_Days,0.824,84%,69%,504,+0.003
Distance_km,0.824,84%,69%,502,+0.003
Transport_Mode,0.828,85%,69%,503,-0.001
Weather is a word the model has never seen ('Typhoon'),0.799,83%,68%,498,+0.028


**What this tells us about noise:**

- **The numbers can be badly off and the model holds.** At 10% noise the scores barely move. Even at 30% noise precision is 82% and recall 70%.
- **Why:** the strongest input is a category, not a measurement. A weather label is either right or wrong, and a wobble in the lead time or the risk score moves the prediction only a little.
- **A wrong weather label is what hurts.** If 1 shipment in 10 carries the wrong weather, ROC AUC falls from 0.83 to 0.80 and precision from 85% to 83%. At 3 in 10 it is 0.76 and 79%. The quality of the weather feed matters more than the precision of any number.

**What this tells us about missing fields:**

- **Four of the six inputs can be blank with almost no loss:** carrier reliability, lead time, distance and transport mode. ROC AUC drops by less than 0.01.
- **A missing risk score makes the model cautious.** It raises about 100 fewer alerts. Precision goes up to 90% and recall falls from 70% to 58%.
- **A missing weather field is the real danger, and the fallback model handles it.** Left to cope, the main model drops to a ROC AUC of 0.68 and precision of 71%. With the fallback model taking over, ROC AUC is 0.80, precision 83% and recall 68%. Most of the loss is recovered.
- **A weather word the model has never seen is handled the same way.** "Typhoon" is not in the training file, so the shipment is routed to the fallback model and nothing breaks.
- **Without the weather and without the lead time, the model cannot work.** ROC AUC falls to 0.60 and recall to 35%.
- **What to do about it:** the app must not show a risk number for a shipment that has neither a weather value nor the three trip fields the fallback needs. It should show "Insufficient data for reliable prediction". The rule is stored in the saved model as `required_any_of`.

**The caveat from notebook 03 applies here too:** the fallback works because the lead time in this file already includes the weather delay. Fed a lead time planned before the weather was known, it would have nothing to read.

## 9. Save the model for the app

The file holds everything the app needs to score a shipment the same way this notebook does: both trained models, the feature lists, the constants inside `add_features`, the alert line, the tier boundaries and the test scores.

In [17]:
import joblib
import sklearn

MODEL_PATH = ROOT / "models" / "aura_final_model.pkl"
MODEL_PATH.parent.mkdir(exist_ok=True)

package = {
    "pipeline": final_model,
    "features": MODEL_FEATURES,
    "numeric_features": MODEL_NUMERIC,
    "categorical_features": MODEL_CATEGORICAL,
    "fallback_pipeline": fallback_model,            # used when the weather is missing or unknown
    "fallback_features": FALLBACK_FEATURES,
    "target": TARGET,
    "weather_severity": WEATHER_SEVERITY,
    "typical_pace": TYPICAL_PACE,
    "high_risk_score": HIGH_RISK_SCORE,
    "low_reliability": LOW_RELIABILITY,
    "alert_line": ALERT_LINE,
    "tier_edges": TIER_EDGES,
    "tier_names": TIER_NAMES,
    # A shipment can be scored if it has the weather, or else everything the fallback needs
    "required_any_of": [["Weather_Condition"], ["Lead_Time_Days", "Distance_km", "Transport_Mode"]],
    "test_scores": {k: round(float(v), 4) for k, v in scorecard["test_score"].items()},
    "trained_on_shipments": int(len(work)),
    "tested_on_shipments": int(len(test)),
    "sklearn_version": sklearn.__version__,
}
joblib.dump(package, MODEL_PATH, compress=3)

reloaded = joblib.load(MODEL_PATH)
same = np.allclose(reloaded["pipeline"].predict_proba(X_test)[:, 1], p_test)
same_fallback = np.allclose(reloaded["fallback_pipeline"].predict_proba(test[FALLBACK_FEATURES])[:, 1], p_test_fallback)
print(f"Saved to {MODEL_PATH.relative_to(ROOT)} ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")
print(f"Reloaded models give identical predictions: {same and same_fallback}")
print("Stored test scores:", reloaded["test_scores"])

Saved to models\aura_final_model.pkl (3 KB)
Reloaded models give identical predictions: True
Stored test scores: {'Precision': 0.8484, 'Recall': 0.7031, 'F1 score': 0.769, 'ROC AUC': 0.8269, 'Average precision': 0.9009}


**Using it:** build the features with `add_features`, then call `package["pipeline"].predict_proba(shipments[package["features"]])[:, 1]` to get the disruption risk for each shipment. If the weather is blank or not one of the five known words, use `package["fallback_pipeline"]` with `package["fallback_features"]` in the same way. The function `predict_risk` in section 2 does both.

The code in `src/` (`schema.py`, `data_loader.py`, `features.py`, `model.py`) was written for the previous training file and has not been changed by these notebooks. Connecting the app to this model means moving `add_features` and `predict_risk` into `src/` and updating the column names there.

## The scorecard

All figures are from the 1,000 test shipments, which played no part in building the model.

| Measure | Result | 95% range |
|---|---|---|
| Precision (alerts that are real) | 85% | 82% to 88% |
| Recall (disruptions caught) | 70% | 67% to 74% |
| F1 score | 0.77 | 0.74 to 0.80 |
| ROC AUC | 0.83 | 0.80 to 0.85 |
| False alarms per 1,000 shipments | 77, against 387 if everything is flagged | |
| Disruption rate in the Critical tier | 97% | |
| Fewer false alarms than the rule of thumb, same workload | 10% | |
| ROC AUC with the weather field missing (fallback model) | 0.80 | |

### Does it help someone make a better decision?

| What a coordinator needs | Evidence in this notebook |
|---|---|
| **Detect:** will it be disrupted? | More than 8 in 10 alerts are real, and 7 in 10 disruptions are caught ahead of time (section 4) |
| **Categorize:** why? | Each alert is driven by the weather, the route or the carrier, and the model shows which (section 6) |
| **Prioritize:** which first? | Risk tiers separate cleanly, from 22% in Low to 97% in Critical (section 5) |
| **Explain:** what is driving it? | Three checkable inputs: weather and pace, route risk score, carrier reliability (section 6) |
| **Decide:** what to do? | The action that fits the cause cuts risk sharply and the others do not. For 44% of alerts the best decision is not to spend (section 7) |

### What this changes in the demo story

The earlier plan was to click *Replenish Dry Ice* and watch the risk fall from 82% to about 11%. **This model cannot show that, and it should not.** In this data the thermal buffer has no connection to whether a disruption happens.

What the model can show honestly:

- Click *Hold or route around the weather* on a storm or hurricane alert, and the risk falls from about 99% to about 41% on average.
- Click *Reroute off the high-risk corridor* on a calm-weather alert, and the risk falls from about 65% to about 35%.
- Click *Replenish Dry Ice*, and the hours of thermal runway go up while the risk stays where it is. The screen should say so.

The categories in the shared data contract (cold-chain excursion, transport delay, handover delay) cannot be produced from this file either, because it does not record what kind of disruption happened. The categories the model can support are the drivers: weather, route risk, carrier, or a combination.

### What the model cannot do

- It misses 3 in 10 disruptions. Most of those come with no warning in any recorded input.
- It cannot call a shipment safe. About 1 in 5 of the shipments it rates Low is still disrupted.
- It cannot say what kind of disruption is coming, or whether the cargo will spoil.
- It is only a little better than a rule that sorts by weather and then by risk score. Its value is the honest percentage, the ranking inside each weather, and the fallback when the weather field is missing.

### Limits we state openly

- The test set has 1,000 shipments, so every score carries the margin of error shown in section 4.
- The file looks generated, with flat spreads and equal shares in every category. The scores describe this dataset. Shipments from a real carrier could score differently, and the model should be re-checked on them before use.
- Only one shipment in five is pharmaceuticals. The buffer, storage standard, cargo value and urgency columns are simulated by us and play no part in the prediction.
- The intervention results show how the model's prediction responds when inputs change. The size of each effect, its cost and the 10% loss share are assumptions, and they should be replaced with a carrier's own figures.
- Say "simulated shipments" in the pitch, never "real data".